# Two-Layer Neural Network from Scratch with NumPy

## What this notebook demonstrates

- neural networks from scratch using NumPy
- affine layers, ReLU activations, and softmax loss
- forward and backward passes with cached intermediate values
- numerical gradient checking
- stochastic gradient descent and alternative optimizers

## Academic context

This notebook is a cleaned portfolio version of MSc coursework and implementation practice. It is included to demonstrate foundational computer vision and deep learning skills.

## Local helper package

This notebook uses the included project-local `engine/` package for coursework helper code such as data loading, classifiers, layers, optimization, and visualization utilities.


In [ ]:
# Project-local setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "engine").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the project root containing the engine/ package.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")


### Intro


In this notebook we will implement fully-connected networks using a modular approach.

For each layer we will implement a `forward` and a `backward` function.

The `forward` function will receive inputs, weights, and other parameters and will return both an output and a `cache` object storing data needed for the backward pass, like this:

```python
def layer_forward(x, w):
  """ Receive inputs x and weights w """
  # Do some computations ...
  z = # ... some intermediate value
  # Do some more computations ...
  out = # the output
   
  cache = (x, w, z, out) # Values we need to compute gradients
   
  return out, cache
```

The backward pass will receive upstream derivatives and the `cache` object, and will return gradients with respect to the inputs and weights, like this:

```python
def layer_backward(dout, cache):
  """
  Receive dout (derivative of loss with respect to outputs) and cache,
  and compute derivative with respect to inputs.
  """
  # Unpack cache values
  x, w, z, out = cache
  
  # Use values in cache to compute derivatives
  dx = # Derivative of loss with respect to x
  dw = # Derivative of loss with respect to w
  
  return dx, dw
```

After implementing a bunch of layers this way, we will be able to easily combine them to build classifiers with different architectures.


### Set up code


In [ ]:
# As usual, a bit of setup
from __future__ import print_function
import time
import numpy as np
import matplotlib.pyplot as plt
from engine.classifiers.two_layer_net import *
from engine.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from engine.solver import Solver

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading external modules
%load_ext autoreload
%autoreload 2

def rel_error(x, y):
  """ returns relative error """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))


### CIFAR-10 Data Loading and Pre-processing


⚠️ **NOTICE**: In case you have already stored CIFAR-10 dataset in ``.npy`` files (as done in ``svm.ipynb``), you can **SKIP** this code snippet.


In [ ]:
from engine.data_utils import load_CIFAR10

# Load the raw CIFAR-10 data.
cifar10_dir = str(DATA_DIR / 'cifar-10-batches-py')

# Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# As a sanity check, we print out the size of the training and test data.
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


### Load .npy files


Load dataset from ``.npy`` files! This way is really faster!


In [ ]:
# Specify adrs dir, so that arrays dir is
# automatically retrieved and arrays can be loaded from it
data_dir = str(DATA_DIR)
arrays_dir = os.path.join(data_dir, 'arrays')

# Load arrays if needed. Uncomment to use!
X_train = np.load(os.path.join(arrays_dir, 'full_X_train.npy'))
y_train = np.load(os.path.join(arrays_dir, 'full_y_train.npy'))
X_test = np.load(os.path.join(arrays_dir, 'full_X_test.npy'))
y_test = np.load(os.path.join(arrays_dir, 'full_y_test.npy'))

# Num of samples for training and test
num_training = X_train.shape[0]
num_test = X_test.shape[0]


### Split data


In [ ]:
# Split the data into train, val, and test sets. In addition we will
# create a small development set as a subset of the training data.
# We can use this for development so our code runs faster.
num_training = 45000
num_validation = 5000
num_test = 10000
num_dev = 500

# Our validation set will be num_validation points from the original
# training set.
mask = range(num_training, num_training + num_validation)
X_val = X_train[mask]
y_val = y_train[mask]

# Our training set will be the first num_train points from the original
# training set.
mask = range(num_training)
X_train = X_train[mask]
y_train = y_train[mask]

# We will also make a development set, which is a small subset of
# the training set.
mask = np.random.choice(num_training, num_dev, replace=False)
X_dev = X_train[mask]
y_dev = y_train[mask]

# We use the first num_test points of the original test set as our
# test set.
mask = range(num_test)
X_test = X_test[mask]
y_test = y_test[mask]

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


### Fully Connected / Linear layer / Affine layer: forward


**Implementation note.**

This section validates the `affine_forward()` function.

The following cell checks the implementation:


In [ ]:
from engine.layers import *
# Test the affine_forward function

num_inputs = 2
input_shape = (4, 5, 6)
output_dim = 3

input_size = num_inputs * np.prod(input_shape)
weight_size = output_dim * np.prod(input_shape)

x = np.linspace(-0.1, 0.5, num=input_size).reshape(num_inputs, *input_shape)
w = np.linspace(-0.2, 0.3, num=weight_size).reshape(np.prod(input_shape), output_dim)
b = np.linspace(-0.3, 0.1, num=output_dim)

out, _ = affine_forward(x, w, b)
correct_out = np.array([[ 1.49834967,  1.70660132,  1.91485297],
                        [ 3.25553199,  3.5141327,   3.77273342]])

# Compare your output with ours. The error should be around e-9 or less.
print('Testing affine_forward function:')
print('difference: ', rel_error(out, correct_out))


### Fully Connected / Linear layer / Affine layer: backward


**Implementation note.**

This section validates the `affine_backward()` implementation in `engine/layers.py` using numeric gradient checking.

**Hint:** For the **weight** and **input** gradient, have a look at [this](http://cs231n.stanford.edu/handouts/linear-backprop.pdf) and [this](https://web.eecs.umich.edu/~justincj/teaching/eecs442/notes/linear-backprop.html).

For the **bias** gradient, have a look at [this](https://datascience.stackexchange.com/questions/20139/gradients-for-bias-terms-in-backpropagation) and [this](https://eli.thegreenplace.net/2018/backpropagation-through-a-fully-connected-layer/).


In [ ]:
# Test the affine_backward function
np.random.seed(231)
x = np.random.randn(10, 2, 3)
w = np.random.randn(6, 5)
b = np.random.randn(5)
dout = np.random.randn(10, 5)

dx_num = eval_numerical_gradient_array(lambda x: affine_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_forward(x, w, b)[0], b, dout)

_, cache = affine_forward(x, w, b)
dx, dw, db = affine_backward(dout, cache)

# The error should be around e-10 or less
print('Testing affine_backward function:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))


### ReLU activation: forward


**Implementation note.**

This section validates the ReLU forward pass in `engine/layers.py`.


In [ ]:
# Test the relu_forward function

x = np.linspace(-0.5, 0.5, num=12).reshape(3, 4)

out, _ = relu_forward(x)
correct_out = np.array([[ 0.,          0.,          0.,          0.,        ],
                        [ 0.,          0.,          0.04545455,  0.13636364,],
                        [ 0.22727273,  0.31818182,  0.40909091,  0.5,       ]])

# Compare your output with ours. The error should be on the order of e-8
print('Testing relu_forward function:')
print('difference: ', rel_error(out, correct_out))


### ReLU activation: backward


**Implementation note.**

This section validates the ReLU backward pass in `engine/layers.py` using numeric gradient checking.

**Hint:** For the **ReLU** gradient, have a look at [this](https://datascience.stackexchange.com/questions/19272/deep-neural-network-backpropogation-with-relu).


In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10)
dout = np.random.randn(*x.shape)

dx_num = eval_numerical_gradient_array(lambda x: relu_forward(x)[0], x, dout)

_, cache = relu_forward(x)
dx = relu_backward(dout, cache)

# The error should be on the order of e-12
print('Testing relu_backward function:')
print('dx error: ', rel_error(dx_num, dx))


**Concept check.**

This notebook focuses on ReLU, but there are a number of different activation functions that one could use in neural networks, each with its pros and cons. In particular, an issue commonly seen with activation functions is getting zero (or close to zero) gradient flow during backpropagation. A problem that is commonly known as the vanishing gradient problem. Which of the following activation functions have this problem?
1. Sigmoid
2. ReLU
3. Leaky ReLU

Provide a brief explanation (analytical or graphical) why is this happening.

What's the minor problem in ReLU that was the reason for designing the Leaky version of it?


**Answer.** 1. Sigmoid

**Explanation.** For values close to 1 or 0 the gradient of the Sigmoid function is close to 0, thus in a backpropagation scheme the gradient will be very small which will lead to the vanishing gradient problem (weights update will be insignificant). ReLU does not have this problem because it doesn't saturate like sigmoid, but, its weakness is that for negative values the gradient will be 0 and this can lead to dead neurons. This is the main reason of introducing Leaky ReLU, which always has a positive gradient (instead of 0 in ReLU), because when the input is negative the gradient is a small constant a.


### "Sandwich" layers / Stacking layers


There are some common patterns of layers that are frequently used in neural nets. For example, affine layers are frequently followed by a ReLU nonlinearity. To make these common patterns easy, we define convenience layers in the file `engine/layer_utils.py`.

Take a look at the `affine_relu_forward()` and `affine_relu_backward()` functions, and run the following to numerically gradient check the backward pass:


In [ ]:
from engine.layer_utils import affine_relu_forward, affine_relu_backward
np.random.seed(231)
x = np.random.randn(2, 3, 4)
w = np.random.randn(12, 10)
b = np.random.randn(10)
dout = np.random.randn(2, 10)

out, cache = affine_relu_forward(x, w, b)
dx, dw, db = affine_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: affine_relu_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_relu_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_relu_forward(x, w, b)[0], b, dout)

# Relative error should be around e-10 or less
print('Testing affine_relu_forward and affine_relu_backward:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))


### Loss: Softmax and SVM


**Implementation note.**

This section implements the loss and gradient for softmax and SVM in the `softmax_loss()` and `svm_loss()` function in `engine/layers.py`.

These implementations mirror the standalone softmax and SVM classifiers in `engine/classifiers/`.

You can make sure that the implementations are correct by running the following:


In [ ]:
np.random.seed(231)
num_classes, num_inputs = 10, 50
x = 0.001 * np.random.randn(num_inputs, num_classes)
y = np.random.randint(num_classes, size=num_inputs)

dx_num = eval_numerical_gradient(lambda x: svm_loss(x, y)[0], x, verbose=False)
loss, dx = svm_loss(x, y)

# Test svm_loss function. Loss should be around 9 and dx error should be around the order of e-9
print('Testing svm_loss:')
print('loss: ', loss)
print('dx error: ', rel_error(dx_num, dx))

dx_num = eval_numerical_gradient(lambda x: softmax_loss(x, y)[0], x, verbose=False)
loss, dx = softmax_loss(x, y)

# Test softmax_loss function. Loss should be close to 2.3 and dx error should be around e-8
print('\nTesting softmax_loss:')
print('loss: ', loss)
print('dx error: ', rel_error(dx_num, dx))


### Two-layer network


**Implementation note.**

This section validates the `TwoLayerNet` class.

Read through it to make sure you understand the API.

The following cell checks the implementation.


In [ ]:
np.random.seed(231)
N, D, H, C = 3, 5, 50, 7
X = np.random.randn(N, D)
y = np.random.randint(C, size=N)

std = 1e-3
model = TwoLayerNet(input_dim=D, hidden_dim=H, num_classes=C, weight_scale=std)

print('Testing initialization ... ')
W1_std = abs(model.params['W1'].std() - std)
b1 = model.params['b1']
W2_std = abs(model.params['W2'].std() - std)
b2 = model.params['b2']
assert W1_std < std / 10, 'First layer weights do not seem right'
assert np.all(b1 == 0), 'First layer biases do not seem right'
assert W2_std < std / 10, 'Second layer weights do not seem right'
assert np.all(b2 == 0), 'Second layer biases do not seem right'

print('Testing test-time forward pass ... ')
model.params['W1'] = np.linspace(-0.7, 0.3, num=D*H).reshape(D, H)
model.params['b1'] = np.linspace(-0.1, 0.9, num=H)
model.params['W2'] = np.linspace(-0.3, 0.4, num=H*C).reshape(H, C)
model.params['b2'] = np.linspace(-0.9, 0.1, num=C)
X = np.linspace(-5.5, 4.5, num=N*D).reshape(D, N).T
scores = model.loss(X)
correct_scores = np.asarray(
  [[11.53165108,  12.2917344,   13.05181771,  13.81190102,  14.57198434, 15.33206765,  16.09215096],
   [12.05769098,  12.74614105,  13.43459113,  14.1230412,   14.81149128, 15.49994135,  16.18839143],
   [12.58373087,  13.20054771,  13.81736455,  14.43418138,  15.05099822, 15.66781506,  16.2846319 ]])
scores_diff = np.abs(scores - correct_scores).sum()
assert scores_diff < 1e-6, 'Problem with test-time forward pass'

print('Testing training loss (no regularization)')
y = np.asarray([0, 5, 1])
loss, grads = model.loss(X, y)
correct_loss = 3.4702243556
assert abs(loss - correct_loss) < 1e-10, 'Problem with training-time loss'

model.reg = 1.0
loss, grads = model.loss(X, y)
correct_loss = 26.5948426952
assert abs(loss - correct_loss) < 1e-10, 'Problem with regularization loss'

# Errors should be around e-7 or less
for reg in [0.0, 0.7]:
  print('Running numeric gradient check with reg = ', reg)
  model.reg = reg
  loss, grads = model.loss(X, y)

  for name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    grad_num = eval_numerical_gradient(f, model.params[name], verbose=False)
    print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))


### Solver


**Implementation note.**

This section uses the training API in `engine/solver.py`.

This section uses the `sgd()` implementation in `engine/optim.py`.

After doing so, use a `Solver` instance to train a `TwoLayerNet` that achieves about `32%` accuracy on the validation set.


In [ ]:
input_size = 32 * 32 * 3
hidden_size = 50
num_classes = 10
model = TwoLayerNet(input_size, hidden_size, num_classes)
solver = None
data = {"X_train":X_train, "y_train":y_train, "X_val":X_val, \
        "y_val":y_val, "X_test":X_test, "y_test":y_test}

# accuracy on the validation set.                                            #
solver = Solver(model, data,
                update_rule='sgd',
                optim_config={
                    'learning_rate': 1e-3,
                },
                lr_decay=0.95,
                num_epochs=10, batch_size=100,
                print_every=100)
solver.train()


### Debug the training


With the default parameters above, the baseline validation accuracy is expected to be modest, around `32%` in the original run.

One strategy for getting insight into what's wrong is to plot the loss function and the accuracies on the training and validation sets during optimization.

Another strategy is to visualize the weights that were learned in the first layer of the network. In most neural networks trained on visual data, the first layer weights typically show some visible structure when visualized.


#### Visualize loss and accuracy


In [ ]:
# Run this cell to visualize training loss and train / val accuracy

plt.subplot(2, 1, 1)
plt.title('Training loss')
plt.plot(solver.loss_history, 'o')
plt.xlabel('Iteration')

plt.subplot(2, 1, 2)
plt.title('Accuracy')
plt.plot(solver.train_acc_history, '-o', label='train')
plt.plot(solver.val_acc_history, '-o', label='val')
plt.plot([0.5] * len(solver.val_acc_history), 'k--')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.gcf().set_size_inches(15, 12)
plt.show()


#### Visualize weights


In [ ]:
from engine.vis_utils import visualize_grid

# Visualize the weights of the network

def show_net_weights(net):
    W1 = net.params['W1']
    W1 = W1.reshape(32, 32, 3, -1).transpose(3, 0, 1, 2)
    plt.imshow(visualize_grid(W1, padding=3).astype('uint8'))
    plt.gca().axis('off')
    plt.show()

show_net_weights(model)


### Tune your hyperparameters


**What's wrong?** Looking at the visualizations above, we see that the loss is decreasing more or less linearly, which seems to suggest that the learning rate may be too low. Moreover, there is no gap between the training and validation accuracy, suggesting that the model we used has low capacity, and that we should increase its size. On the other hand, with a very large model we would expect to see more overfitting, which would manifest itself as a very large gap between the training and validation accuracy.

**Tuning**. Tuning the hyperparameters and developing intuition for how they affect the final performance is a large part of using Neural Networks, so the goal is to get a lot of practice. Below, experiment with different values of the various hyperparameters, including hidden layer size, learning rate, number of training epochs, and regularization strength. You might also consider tuning the learning rate decay, but the model can get good performance using the default value.

**Approximate results**. The original target was a classification accuracy of greater than 40% on the validation set. Our best network gets over `46%` on the validation set.


**Implementation note.**

Your goal in this notebook is to achieve the highest possible validation accuracy on CIFAR-10 using a fully-connected neural network, with `46%` serving as a strong reference target.

To accomplish this, implement a hyperparameter tuning process, trying different configurations of learning rates, hidden layer sizes, and other strategies you think may improve performance.

Here are some ideas to explore:

•	**Hyperparameter Tuning**: Experiment with various learning rates, hidden layer sizes, and batch sizes to find the optimal configuration.

•	**Dimensionality Reduction**: Consider using PCA to reduce the input dimensionality, which can help improve performance by simplifying the feature space.

•	**Regularization Techniques**: Implement dropout or L2 regularization to prevent overfitting.

Feel free to implement your own ideas, write new functions, or adapt existing ones to meet the target validation accuracy!


In [ ]:
# WARNING: THIS WILL TAKE 2-3 HOURS.


import numpy as np
import cv2
from skimage.filters import gabor

# Convert RGB to grayscale as HoG works on single-channel images
def rgb_to_grayscale(image):
    # Use OpenCV to convert RGB to grayscale
    return cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

def extract_gabor_features(image, frequencies, orientations):
    # Ensure the image is in grayscale
    if len(image.shape) == 3:
        if image.dtype != np.uint8:
            image = (image * 255).astype(np.uint8)  # Normalize and convert to uint8
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    elif image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)  # Convert grayscale float64 to uint8

    features = []
    for freq in frequencies:
        for theta in orientations:
            # Apply Gabor filter
            real, imag = gabor(image, frequency=freq, theta=theta)

            # Compute statistics
            mean_real = np.mean(real)
            std_real = np.std(real)
            mean_imag = np.mean(imag)
            std_imag = np.std(imag)

            # Append features to the list
            features.extend([mean_real, std_real, mean_imag, std_imag])

    return np.array(features)


# Define Gabor filter parameters
frequencies = [0.1, 0.2, 0.3]  # Adjust based on your use case
orientations = [0, np.pi/4, np.pi/2, 3*np.pi/4]  # 0°, 45°, 90°, 135°

X_train_gabor = []
X_val_gabor = []
X_test_gabor = []
for image in X_train:
    features = extract_gabor_features(image, frequencies, orientations)
    X_train_gabor.append(features)
del X_train

for image in X_val:
    features = extract_gabor_features(image, frequencies, orientations)
    X_val_gabor.append(features)
del X_val

for image in X_test:
    features = extract_gabor_features(image, frequencies, orientations)
    X_test_gabor.append(features)
del X_test

X_train_gabor = np.array(X_train_gabor)
X_val_gabor = np.array(X_val_gabor)
X_test_gabor = np.array(X_test_gabor)


In [ ]:
best_model = None

# model in best_model.                                                          #
#                                                                               #
# To help debug your network, it may help to use visualizations similar to the  #
# ones we used above; these visualizations will have significant qualitative    #
# differences from the ones we saw above for the poorly tuned network.          #
#                                                                               #
# Tweaking hyperparameters by hand can be fun, but you might find it useful to  #
# write code to sweep through possible combinations of hyperparameters          #
# automatically like we did on the previous notebooks.                          #

"""
Ideas:
    - Use an image descriptor like HoG.
    - Tune lr, lr_decay, batch_size, hidden_size and reg.
    - Try a different weights initialization method
"""

# model
input_size = 48
hidden_size = 600
num_classes = 10
model = TwoLayerNet(input_size, hidden_size, num_classes, reg=1e-2)
solver = None
data = {"X_train":X_train_gabor, "y_train":y_train, "X_val":X_val_gabor, \
        "y_val":y_val, "X_test":X_test_gabor, "y_test":y_test}

solver = Solver(model, data,
                update_rule='sgd',
                optim_config={
                    'learning_rate': 0.001,
                },
                lr_decay=0.99,
                num_epochs=50, batch_size=64,
                print_every=15000)
solver.train()


In [ ]:
best_model = model


### Test your model!


Run the selected model on the validation and test sets and report the observed accuracies.


In [ ]:
print(best_model)
y_val_pred = np.argmax(best_model.loss(data['X_val']), axis=1)
print('Validation set accuracy: ', (y_val_pred == data['y_val']).mean())


In [ ]:
y_test_pred = np.argmax(best_model.loss(data['X_test']), axis=1)
print('Test set accuracy: ', (y_test_pred == data['y_test']).mean())


### Try alternative Optimizers


**Implementation note.**

Now that you’ve optimized your model using vanilla `SGD`, let’s explore the effects of different optimization techniques.

This section compares the `sgd_momentum()`, `rmsprop()`, and `adam()` optimizers implemented in `engine/optim.py`.

After implementing these functions, use a Solver instance to train a TwoLayerNet with each of these alternative optimizers. Observe and compare the performance of each optimizer in terms of validation accuracy and training speed.


In [ ]:
best_model = None


# RMSProp, and Adam optimizers. Observe and compare the performance of each     #
# optimizer in terms of validation accuracy and training speed.                 #
# model
input_size = 48
hidden_size = 200  # 120
num_classes = 10

model = TwoLayerNet(input_size, hidden_size, num_classes, reg=1e-2)
solver = None
data = {"X_train":X_train_gabor, "y_train":y_train, "X_val":X_val_gabor, \
        "y_val":y_val, "X_test":X_test_gabor, "y_test":y_test}

mt = ['sgd_momentum', 'rmsprop', 'adam']
for m in mt:
    print(f'Method: {m}')
    solver = Solver(model, data,
                    update_rule=m,
                    optim_config={
                        'learning_rate': 0.00001,  # 0.001
                    },
                    lr_decay=0.98,
                    num_epochs=20, batch_size=128, # epochs 50
                    print_every=15000)
    solver.train()


**Concept check.**

Now that you have trained a Neural Network, you may find that your testing accuracy is much lower than the training accuracy. In what ways can we decrease this gap? Select all that apply.

1. Train on a larger dataset.
2. Add more hidden units.
3. Increase the regularization strength.
4. None of the above.

**Answer.** 1 & 3 are correct.

**Explanation.** Training on a larger dataset helps the model generalize better. Regularization (l2 or dropout) encourages smaller weights aka simpler model that generalizes better.
